# Soil Health Intelligence — GPU Training
**Krishna District, Andhra Pradesh** · CatBoost + MAPIE Conformal Prediction

**Before running:** Runtime → Change runtime type → T4 GPU

**Steps:**
1. Run Cell 1 — install deps
2. Run Cell 2 — upload `combined_training_data.csv`
3. Run Cell 3 — train all 10 soil parameters (~15 min on GPU)
4. Run Cell 4 — download `models.zip`

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q catboost mapie loguru tqdm scikit-learn

In [ ]:
# ── Cell 2: Upload training data ───────────────────────────────────────────────
from google.colab import files
print('Upload combined_training_data.csv from:')
print('  data/processed/combined_training_data.csv')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# ── Cell 3: CPU Training (skip already-trained params) ────────────────────────
import json, warnings, os, glob
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

warnings.filterwarnings('ignore')

SOIL_PARAMS = ['pH', 'EC', 'OC', 'N', 'P', 'K', 'Fe', 'Cu', 'B', 'Zn']
MODELS_DIR  = Path('models')
MODELS_DIR.mkdir(exist_ok=True)

# ── Skip params that already have a saved meta file ───────────────────────────
already_done = set()
for f in MODELS_DIR.glob('*_combined_*_meta.json'):
    param = f.name.split('_')[0]
    already_done.add(param)

TODO = [p for p in SOIL_PARAMS if p not in already_done]
print(f'Already done: {sorted(already_done)}')
print(f'To train:     {TODO}')

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('combined_training_data.csv')
if 'source' in df.columns:
    df = df.drop(columns=['source'])
print(f'Loaded: {len(df):,} rows x {len(df.columns)} columns')

feature_cols = [
    c for c in df.columns
    if not c.startswith('label_') and c not in
    ['lat', 'lon', 'farmer_id', 'village', 'mandal', 'district', 'sample_date', 'geometry_wkt']
]
print(f'Features: {len(feature_cols)}')

# ── Helpers ────────────────────────────────────────────────────────────────────
def spatial_kfold(data, n_folds=3):
    from sklearn.cluster import KMeans
    coords = data[['lat', 'lon']].values
    blocks = KMeans(n_clusters=n_folds, n_init=10, random_state=42).fit_predict(coords)
    return [(np.where(blocks != f)[0], np.where(blocks == f)[0]) for f in range(n_folds)]

def build_mapie(X_tr, y_tr):
    from catboost import CatBoostRegressor
    from mapie.regression import CrossConformalRegressor
    base = CatBoostRegressor(
        iterations=500, learning_rate=0.07, depth=6,
        task_type='CPU', random_seed=42, verbose=0,
    )
    model = CrossConformalRegressor(estimator=base, confidence_level=0.90, method='plus', cv=3, n_jobs=-1)
    model.fit_conformalize(X_tr, y_tr)
    return model

def build_qrf(X_tr, y_tr):
    from sklearn.ensemble import GradientBoostingRegressor
    quantiles = {}
    for q, a in [('low', 0.05), ('mid', 0.50), ('high', 0.95)]:
        quantiles[q] = GradientBoostingRegressor(
            loss='quantile', alpha=a, n_estimators=150,
            max_depth=4, learning_rate=0.08, random_state=42
        ).fit(X_tr, y_tr)
    return quantiles

def calc_metrics(y_true, y_pred, y_lo=None, y_hi=None):
    from sklearn.metrics import mean_squared_error, r2_score
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2   = float(r2_score(y_true, y_pred))
    m    = {'RMSE': round(rmse, 4), 'R2': round(r2, 4)}
    if y_lo is not None:
        cov = float(np.mean((y_true >= y_lo) & (y_true <= y_hi)))
        m['coverage_90']       = round(cov, 4)
        m['calibration_error'] = round(abs(cov - 0.90), 4)
    return m

def avg_folds(fold_list):
    if not fold_list:
        return {}
    keys = fold_list[0].keys()
    return {key: round(float(np.mean([f[key] for f in fold_list])), 4) for key in keys}

# ── Training loop ──────────────────────────────────────────────────────────────
summary = []

for param in TODO:
    label_col = f'label_{param}'
    if label_col not in df.columns:
        print(f'[SKIP] {param} — no label column')
        continue

    sub = df[feature_cols + [label_col, 'lat', 'lon']].dropna()
    if len(sub) < 30:
        print(f'[SKIP] {param} — only {len(sub)} samples')
        continue

    X = sub[feature_cols].values.astype(np.float32)
    y = sub[label_col].values.astype(np.float32)
    print(f'\n=== {param}: {len(sub):,} samples ===')

    splits   = spatial_kfold(sub)
    cv_mapie = []
    cv_qrf   = []

    for fold_i, (tr_idx, val_idx) in enumerate(splits):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        if len(X_val) == 0:
            continue
        try:
            m = build_mapie(X_tr, y_tr)
            y_pred, ivs = m.predict_interval(X_val)
            cv_mapie.append(calc_metrics(y_val, y_pred, ivs[:, 0, 0], ivs[:, 1, 0]))
        except Exception as e:
            print(f'  MAPIE fold {fold_i} failed: {e}')
        try:
            q = build_qrf(X_tr, y_tr)
            cv_qrf.append(calc_metrics(y_val, q['mid'].predict(X_val),
                                       q['low'].predict(X_val), q['high'].predict(X_val)))
        except Exception as e:
            print(f'  QRF fold {fold_i} failed: {e}')

    avg_mapie = avg_folds(cv_mapie)
    avg_qrf   = avg_folds(cv_qrf)

    for name, m in [('MAPIE', avg_mapie), ('QRF', avg_qrf)]:
        if m:
            print(f'  {name}  RMSE={m.get("RMSE","?"):.4f}  R2={m.get("R2","?"):.3f}  '
                  f'Cov={m.get("coverage_90", 0):.2%}  CalErr={m.get("calibration_error", 0):.4f}')

    w = {}
    for name, m in [('mapie', avg_mapie), ('qrf', avg_qrf)]:
        w[name] = 1.0 / (m.get('calibration_error', 1.0) + 1e-6)
    total = sum(w.values())
    w = {k: round(v / total, 3) for k, v in w.items()}
    print(f'  Weights: {w}')

    print(f'  Final fit on {len(X):,} samples…')
    final_mapie, final_qrf = None, None
    try:
        final_mapie = build_mapie(X, y)
    except Exception as e:
        print(f'  Final MAPIE failed: {e}')
    try:
        final_qrf = build_qrf(X, y)
    except Exception as e:
        print(f'  Final QRF failed: {e}')

    best_rmse = avg_mapie.get('RMSE', 9999)
    best_r2   = avg_mapie.get('R2', 0)
    stem      = f'{param}_combined_rmse{best_rmse:.3f}_r2{best_r2:.3f}'

    if final_mapie:
        joblib.dump(final_mapie, MODELS_DIR / f'{stem}_mapie.pkl')
    if final_qrf:
        joblib.dump(final_qrf, MODELS_DIR / f'{stem}_qrf.pkl')

    meta = {
        'param': param, 'window': 'combined', 'n_samples': len(sub),
        'feature_cols': feature_cols,
        'cv_metrics': {'mapie': avg_mapie, 'qrf': avg_qrf},
        'ensemble_weights': w,
    }
    with open(MODELS_DIR / f'{stem}_meta.json', 'w') as fh:
        json.dump(meta, fh, indent=2)

    print(f'  Saved {stem}')
    summary.append({'param': param, 'RMSE': best_rmse, 'R2': best_r2,
                    'coverage': avg_mapie.get('coverage_90', 0)})

print('\n=== Summary ===')
print(f'{"Param":6s}  {"RMSE":>8s}  {"R2":>6s}  {"Coverage":>10s}')
for s in summary:
    print(f'{s["param"]:6s}  {s["RMSE"]:>8.4f}  {s["R2"]:>6.3f}  {s["coverage"]:>10.2%}')
print(f'\nModels in: {MODELS_DIR.resolve()}')

In [ ]:
# ── Cell 4: Download all models as a zip ──────────────────────────────────────
import shutil
from google.colab import files

zip_path = 'soil_models_combined.zip'
shutil.make_archive('soil_models_combined', 'zip', 'models')
print(f'Zip size: {Path(zip_path).stat().st_size / 1e6:.1f} MB')
files.download(zip_path)
print('Download started — save to your Downloads folder')

## After downloading

Extract `soil_models_combined.zip` and copy all files into:
```
data/models/
```
Then run prediction maps:
```powershell
cd pipeline
python 05_predict_maps.py --combined --all-params --no-gpr --downsample 3
```